In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Techniques d'atténuation et de suppression des erreurs

> **Note:** La version bêta d'un nouveau modèle d'exécution est maintenant disponible. Le modèle d'exécution dirigée offre plus de flexibilité pour personnaliser ton workflow d'atténuation des erreurs. Consulte le guide [Modèle d'exécution dirigée](/guides/directed-execution-model) pour plus d'informations.


<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les prérequis suivants.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Les techniques d'atténuation et de suppression des erreurs sont utilisées pour améliorer la qualité des résultats lors du passage à des charges de travail plus importantes. Cette page fournit des explications de haut niveau sur les techniques de suppression et d'atténuation des erreurs disponibles via Qiskit Runtime.

La cellule suivante importe la primitive Estimator et crée un backend qui sera utilisé pour initialiser l'Estimator dans les cellules de code ultérieures.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Découplage dynamique
Les circuits quantiques sont exécutés sur le matériel IBM&reg; sous forme de séquences d'impulsions micro-ondes qui doivent être planifiées et exécutées à des intervalles de temps précis.
Malheureusement, des interactions indésirables entre les qubits peuvent entraîner des erreurs cohérentes sur les qubits en attente. Le découplage dynamique fonctionne en insérant des séquences d'impulsions sur les qubits en attente pour annuler approximativement l'effet de ces erreurs. Chaque séquence d'impulsions insérée correspond à une opération identité, mais la présence physique des impulsions a pour effet de supprimer les erreurs.
Il existe de nombreux choix possibles de séquences d'impulsions, et la question de quelle séquence est la meilleure pour chaque cas particulier reste un [domaine de recherche actif](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Note que le découplage dynamique est principalement utile pour les circuits contenant des lacunes dans lesquelles certains qubits restent inactifs sans aucune opération les concernant. Si les opérations du circuit sont très densément regroupées, de sorte que tous les qubits sont occupés la plupart du temps, alors l'ajout d'impulsions de découplage dynamique pourrait ne pas améliorer les performances. En fait, cela pourrait même les dégrader en raison des imperfections des impulsions elles-mêmes.

Le schéma ci-dessous représente le découplage dynamique avec une séquence d'impulsions XX. Le circuit abstrait à gauche est mappé sur un calendrier d'impulsions micro-ondes en haut à droite. En bas à droite, on voit le même calendrier, mais avec une séquence de deux impulsions X insérées pendant une période d'inactivité du premier qubit.

![Représentation du découplage dynamique](../docs/images/guides/error-mitigation-explanation/dd.avif)

Le découplage dynamique peut être activé en définissant `enable` sur `True` dans les [options de découplage dynamique](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). L'option `sequence_type` peut être utilisée pour choisir parmi plusieurs séquences d'impulsions différentes. Le type de séquence par défaut est `"XX"`.

La cellule de code suivante montre comment activer le découplage dynamique pour l'Estimator et choisir une séquence de découplage dynamique.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Twirling de Pauli
Le twirling, également connu sous le nom de [compilation aléatoire](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), est une technique largement utilisée pour convertir des canaux de bruit arbitraires en canaux de bruit avec une structure plus spécifique.

Le twirling de Pauli est un type particulier de twirling qui utilise des opérations de Pauli. Il a pour effet de transformer tout canal quantique en un canal de Pauli. Appliqué seul, il peut atténuer le bruit cohérent car ce dernier tend à s'accumuler de manière quadratique avec le nombre d'opérations, tandis que le bruit de Pauli s'accumule linéairement. Le twirling de Pauli est souvent combiné avec d'autres techniques d'atténuation des erreurs qui fonctionnent mieux avec le bruit de Pauli qu'avec un bruit arbitraire.

Le twirling de Pauli est mis en œuvre en encadrant un ensemble de portes choisi avec des portes de Pauli à un qubit choisies aléatoirement, de manière à ce que l'effet idéal de la porte reste le même. Le résultat est qu'un seul circuit est remplacé par un ensemble aléatoire de circuits, tous ayant le même effet idéal. Lors de l'échantillonnage du circuit, les échantillons sont tirés de plusieurs instances aléatoires, plutôt que d'une seule.

![Représentation du twirling de Pauli](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Étant donné que la plupart des erreurs dans le matériel quantique actuel proviennent des portes à deux qubits, cette technique est souvent appliquée exclusivement aux portes à deux qubits (natives). Le schéma suivant représente quelques twirls de Pauli pour les portes CNOT et ECR. Chaque circuit dans une ligne a le même effet idéal.

![Représentation des twirls de portes](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Le twirling de Pauli peut être activé en définissant `enable_gates` sur `True` dans les [options de twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Les autres options notables incluent :

- `num_randomizations` : Le nombre d'instances de circuit à tirer de l'ensemble des circuits twirled.
- `shots_per_randomization` : Le nombre de shots à échantillonner à partir de chaque instance de circuit.

La cellule de code suivante montre comment activer le twirling de Pauli et définir ces options pour l'estimator. Aucune de ces options n'est requise d'être définie explicitement.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Extinction des erreurs de lecture par twirling (TREX)
L'[extinction des erreurs de lecture par twirling (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) atténue l'effet des erreurs de mesure pour l'estimation des valeurs d'espérance des observables de Pauli.
Elle est basée sur la notion de mesures twirled, qui sont réalisées en substituant aléatoirement les portes de mesure par une séquence de (1) une porte Pauli X, (2) une mesure, et (3) un flip de bit classique. Tout comme dans le twirling de portes standard, cette séquence est équivalente à une mesure simple en l'absence de bruit, comme représenté dans le schéma suivant :

![Représentation du twirling de mesure](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

En présence d'erreurs de lecture, le twirling de mesure a pour effet de diagonaliser la matrice de transfert des erreurs de lecture, ce qui la rend facile à inverser. L'estimation de la matrice de transfert des erreurs de lecture nécessite l'exécution de circuits d'étalonnage supplémentaires, ce qui introduit un faible surcoût.

TREX peut être activé en définissant `measure_mitigation` sur `True` dans les [options de résilience de Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pour l'Estimator. Les options d'apprentissage du bruit de mesure sont décrites [ici](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Comme pour le twirling de portes, tu peux définir le nombre de randomisations de circuits et le nombre de shots par randomisation.

La cellule de code suivante montre comment activer TREX et définir ces options pour l'estimator. Aucune de ces options n'est requise d'être définie explicitement.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Extrapolation zéro-bruit (ZNE)
L'extrapolation zéro-bruit (ZNE) est une technique pour atténuer les erreurs dans l'estimation des valeurs d'espérance des observables. Bien qu'elle améliore souvent les résultats, elle ne garantit pas de produire un résultat non biaisé.

La ZNE se compose de deux étapes :

1. _Amplification du bruit_ : Le circuit quantique original est exécuté plusieurs fois à différents niveaux de bruit.
2. _Extrapolation_ : Le résultat idéal est estimé en extrapolant les résultats de valeur d'espérance bruités jusqu'à la limite zéro-bruit.

Les deux étapes d'amplification du bruit et d'extrapolation peuvent être mises en œuvre de nombreuses façons différentes. Qiskit Runtime met en œuvre l'amplification du bruit par « repliement de porte numérique », ce qui signifie que les portes à deux qubits sont remplacées par des séquences équivalentes de la porte et de son inverse. Par exemple, remplacer un unitaire $U$ par $U U^\dagger U$ donnerait un facteur d'amplification du bruit de 3. Pour l'extrapolation, tu peux choisir parmi plusieurs formes fonctionnelles, notamment un ajustement linéaire ou un ajustement exponentiel.
L'image ci-dessous représente le repliement de porte numérique à gauche, et la procédure d'extrapolation à droite.

![Représentation de la ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

La ZNE peut être activée en définissant `zne_mitigation` sur `True` dans les [options de résilience de Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pour l'Estimator.
Les options de Qiskit Runtime pour la ZNE sont décrites [ici](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Les options suivantes sont notables :

- `noise_factors` : Les facteurs de bruit à utiliser pour l'amplification du bruit.
- `extrapolator` : La forme fonctionnelle à utiliser pour l'extrapolation.

La cellule de code suivante montre comment activer la ZNE et définir ces options pour l'estimator. Aucune de ces options n'est requise d'être définie explicitement.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Amplification probabiliste des erreurs (PEA)
L'un des principaux défis de la ZNE est d'amplifier avec précision le bruit affectant le circuit cible. Le repliement de porte offre un moyen simple d'effectuer cette amplification, mais peut être imprécis et conduire à des résultats incorrects. Consulte l'article [« Scalable error mitigation for noisy quantum circuits produces competitive expectation values »](https://arxiv.org/pdf/2108.09197), et en particulier la page 4 des informations supplémentaires pour plus de détails. L'amplification probabiliste des erreurs offre une approche plus précise de l'amplification des erreurs grâce à l'apprentissage du bruit.

La PEA est une technique plus sophistiquée qui effectue des expériences préliminaires pour reconstruire le bruit, puis utilise ces informations pour effectuer une amplification précise. Elle commence par apprendre le modèle de bruit twirled de chaque couche de portes d'intrication dans le circuit avant leur exécution (voir [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) pour les options d'apprentissage pertinentes). Après la phase d'apprentissage, les circuits sont exécutés à chaque facteur de bruit, où chaque couche d'intrication des circuits est amplifiée en injectant de manière probabiliste un bruit à un qubit proportionnel au modèle de bruit appris correspondant. Consulte l'article [« Evidence for the utility of quantum computing before fault tolerance »](https://www.nature.com/articles/s41586-023-06096-3) pour plus de détails.

La PEA se compose de trois étapes :
1. _Apprentissage_ : Le modèle de bruit twirled de chaque couche de portes d'intrication dans le circuit est appris.
1. _Amplification du bruit_ : Le circuit quantique original est exécuté plusieurs fois à différents facteurs de bruit.
2. _Extrapolation_ : Le résultat idéal est estimé en extrapolant les résultats de valeur d'espérance bruités jusqu'à la limite zéro-bruit.

Pour les expériences à l'échelle utilitaire, la PEA est souvent le meilleur choix.

Étant donné que la PEA est une technique d'amplification du bruit ZNE, tu dois également activer la ZNE en définissant `resilience.zne_mitigation = True`. Les autres options [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) peuvent également être utilisées pour définir des extrapolateurs, des niveaux d'amplification, etc. La PEA nécessite un modèle de bruit, qui est automatiquement généré lors de l'utilisation des primitives.

L'extrait de code suivant fournit un exemple où la PEA est utilisée pour atténuer le résultat d'un job Estimator :

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Annulation probabiliste des erreurs (PEC)
L'annulation probabiliste des erreurs (PEC) est une technique pour atténuer les erreurs dans l'estimation des valeurs d'espérance des observables. Contrairement à la ZNE, elle renvoie une estimation non biaisée de la valeur d'espérance. Cependant, elle entraîne généralement un surcoût plus important.

Dans la PEC, l'effet d'un circuit cible idéal est exprimé comme une combinaison linéaire de circuits bruités réellement implémentables en pratique :

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

La sortie du circuit idéal peut alors être reproduite en exécutant différentes instances de circuits bruités tirées d'un ensemble aléatoire défini par la combinaison linéaire. Si les coefficients $\eta_i$ forment une distribution de probabilité, ils peuvent être utilisés directement comme probabilités de l'ensemble. En pratique, certains coefficients sont négatifs, ils forment donc une distribution de quasi-probabilité à la place. Ils peuvent toujours être utilisés pour définir un ensemble aléatoire, mais il y a un surcoût d'échantillonnage lié à la négativité de la distribution de quasi-probabilité, qui est caractérisé par la quantité

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

Le surcoût d'échantillonnage est un facteur multiplicatif sur le nombre de shots nécessaires pour estimer une valeur d'espérance avec une précision donnée, par rapport au nombre de shots qui seraient nécessaires avec le circuit idéal. Il évolue de manière quadratique avec $\gamma$, qui lui-même évolue de manière exponentielle avec la profondeur du circuit.

La PEC peut être activée en définissant `pec_mitigation` sur `True` dans les [options de résilience de Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) pour l'Estimator.
Les options de Qiskit Runtime pour la PEC sont décrites [ici](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Une limite sur le surcoût d'échantillonnage peut être définie à l'aide de l'option `max_overhead`. Note que limiter le surcoût d'échantillonnage peut amener la précision du résultat à dépasser la précision demandée. La valeur par défaut de `max_overhead` est 100.

La cellule de code suivante montre comment activer la PEC et définir l'option `max_overhead` pour l'estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Étapes suivantes
- Consulte le [tutoriel](/tutorials/combine-error-mitigation-techniques) sur la combinaison des options d'atténuation des erreurs avec la primitive Estimator.
- [Configurer l'atténuation des erreurs.](configure-error-mitigation)
- [Configurer la suppression des erreurs.](configure-error-suppression)
- Explorer d'autres [options](runtime-options-overview) pour les primitives Qiskit Runtime.
- Décider dans quel [mode d'exécution](execution-modes) exécuter ton job.